# ViT Memory Autoencoder Training (Colab)

This notebook runs the rebuilt module-based pipeline using `train.py` and `evaluate.py`.

In [ ]:
# Clone repository and install dependencies
import os

repo_dir = "/content/explainable-anomaly-detection"
if not os.path.isdir(repo_dir):
    !git clone https://github.com/eftekin/explainable-anomaly-detection {repo_dir}

%cd /content/explainable-anomaly-detection
!pip install -q -r requirements.txt
!pip install -q kaggle
print("Repository ready.")

In [ ]:
import os
import json

# Kaggle credentials
KAGGLE_USERNAME = "eftekin"
KAGGLE_KEY = ""   # empty — user fills this in

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
cred_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(cred_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(cred_path, 0o600)
print('Kaggle credentials saved to ~/.kaggle/kaggle.json')

In [ ]:
import os
import shutil
import subprocess

project_root = '/content/explainable-anomaly-detection'
if os.path.isdir(project_root):
    os.chdir(project_root)

mvtec_root = os.path.join('data', 'mvtec')
categories = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

if not os.path.exists(mvtec_root):
    os.makedirs(mvtec_root, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'ipythonx/mvtec-ad',
        '-p', '/content',
        '--unzip'
    ], check=True)
    for cat in categories:
        src = os.path.join('/content', cat)
        dst = os.path.join(mvtec_root, cat)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.move(src, dst)

print('Dataset root:', mvtec_root)

In [ ]:
# Run training
!python train.py --data-root data/mvtec --category bottle

In [ ]:
# Run evaluation
!python evaluate.py --data-root data/mvtec --category bottle --checkpoint checkpoints/best_model.pth

## Visualization

This section loads the trained checkpoint and shows:

1. Original image
2. Reconstruction
3. Pixel-wise anomaly map
4. Overlay heatmap (with GT contour when available)

It also plots image-level anomaly score distributions for normal vs defect samples.

In [ ]:
# Generate visualizations from the trained checkpoint
!python visualize.py \
  --data-root data/mvtec \
  --category bottle \
  --checkpoint checkpoints/best_model.pth \
  --output-path outputs \
  --num-per-class 3

# Show generated figures inline
from IPython.display import Image, display

display(Image(filename="outputs/visualization_examples.png"))
display(Image(filename="outputs/visualization_score_distribution.png"))